# Module 08 · Unit 01
# Divisibility and Modular Arithmetic

| | |
|---|---|
| **Estimated time** | 60–70 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Cryptographic Structure \[CS\] |
| **Refresher** | Module 00 · Unit 01 — Proof techniques (proof by contradiction) |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Apply divisibility rules and state the **Division Algorithm**
2. Compute the **GCD** of two integers using the Euclidean Algorithm
3. Perform arithmetic in $\mathbb{Z}_n$ — modular addition, multiplication, and exponentiation
4. Determine when a modular inverse exists and compute it using the **Extended Euclidean Algorithm**
5. State **Fermat's Little Theorem** and apply it to fast modular exponentiation
6. Prove uniqueness of modular inverses using proof by contradiction (fulfilling the Module 00 promise)


## 🔣 Symbol Primer

Number theory introduces compact notation for divisibility and modular relationships.

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $a \mid b$ | Divides | "$a$ divides $b$" | $b = ka$ for some integer $k$ |
| $a \nmid b$ | Does not divide | "$a$ does not divide $b$" | No integer $k$ satisfies $b = ka$ |
| $\gcd(a, b)$ | GCD | "greatest common divisor of $a$ and $b$" | Largest integer dividing both $a$ and $b$ |
| $a \equiv b \pmod{n}$ | Congruence | "$a$ is congruent to $b$ mod $n$" | $n \mid (a - b)$ |
| $\mathbb{Z}_n$ | Integers mod $n$ | "Z sub $n$" | $\{0, 1, 2, \ldots, n-1\}$ with arithmetic mod $n$ |
| $a^{-1} \pmod{n}$ | Modular inverse | "$a$ inverse mod $n$" | The $x$ such that $ax \equiv 1 \pmod{n}$ |
| $\mathbb{Z}_n^*$ | Units mod $n$ | "Z sub $n$ star" | Elements of $\mathbb{Z}_n$ that have a multiplicative inverse |
| $p$ | Prime | "prime" | An integer $> 1$ with exactly two divisors: $1$ and itself |

> **Why this module matters now.** Everything in Module 03 Unit 03 (bijections,
> encryption as a permutation) assumed that decryption worked — that an inverse
> existed. This module explains *when* inverses exist and *how to compute them*.
> That is the mathematical foundation that makes RSA possible.

---


## 1 · Divisibility and the Division Algorithm

We say $a$ **divides** $b$, written $a \mid b$, if there exists an integer $k$
such that $b = ka$. The integer $a$ is called a **divisor** (or **factor**) of $b$.

### Basic Properties

$$a \mid b \wedge b \mid c \;\Rightarrow\; a \mid c \qquad \text{(transitivity)}$$
$$a \mid b \wedge a \mid c \;\Rightarrow\; a \mid (bx + cy) \quad \forall x, y \in \mathbb{Z} \qquad \text{(linearity)}$$

The linearity property is the one used in proofs — it says that any linear
combination of multiples of $a$ is also a multiple of $a$.

### The Division Algorithm

**Theorem.** For any integers $a$ and $b > 0$, there exist unique integers
$q$ (quotient) and $r$ (remainder) such that:

$$a = qb + r \qquad 0 \leq r < b$$

The remainder $r$ is written $a \bmod b$.

**Cryptographic significance.** All modular arithmetic is built on the division
algorithm. When we write $a \equiv r \pmod{b}$, we are saying $a$ and $r$ produce
the same remainder when divided by $b$ — they are equivalent for modular purposes.

### Primes and Composites

A **prime** $p > 1$ has exactly two positive divisors: $1$ and $p$ itself.
Every integer $> 1$ factors uniquely into primes (Fundamental Theorem of Arithmetic).

**Security relevance.** RSA security rests on the hardness of factoring large
composite numbers back into their prime factors. Multiplying two 2048-bit primes
takes milliseconds; factoring the result is believed to require astronomical time.


## 2 · GCD and the Euclidean Algorithm

The **greatest common divisor** $\gcd(a, b)$ is the largest positive integer
dividing both $a$ and $b$.

### Key Properties

$$\gcd(a, b) = \gcd(b, a \bmod b) \qquad \text{(Euclidean reduction)}$$
$$\gcd(a, 0) = a \qquad \text{(base case)}$$

The first identity is the engine of the Euclidean Algorithm:

### Euclidean Algorithm

```
gcd(a, b):
  while b ≠ 0:
    a, b = b, a mod b
  return a
```

**Example:** $\gcd(252, 105)$

$$252 = 2 \times 105 + 42$$
$$105 = 2 \times 42 + 21$$
$$42 = 2 \times 21 + 0$$

So $\gcd(252, 105) = 21$.

### Bézout's Identity

**Theorem.** For any integers $a$ and $b$ with $d = \gcd(a,b)$, there exist
integers $s$ and $t$ such that:

$$as + bt = d$$

The coefficients $s$ and $t$ are the **Bézout coefficients** — computed by
the Extended Euclidean Algorithm (Section 4). This identity is the foundation
of modular inverse computation: if $\gcd(a, n) = 1$, then $as + nt = 1$,
so $as \equiv 1 \pmod{n}$ — meaning $s$ is the modular inverse of $a$.

### Coprimality

Two integers are **coprime** (or **relatively prime**) if $\gcd(a, b) = 1$.
This is the condition for a modular inverse to exist:

$$a^{-1} \pmod{n} \text{ exists} \;\Longleftrightarrow\; \gcd(a, n) = 1$$


## 3 · Modular Arithmetic

### Congruence

$a \equiv b \pmod{n}$ *(a is congruent to b mod n)* means $n \mid (a - b)$.
Equivalently, $a$ and $b$ have the same remainder when divided by $n$.

Congruence is an equivalence relation (Module 03 · Unit 02): reflexive
($a \equiv a$), symmetric, and transitive. The equivalence classes are the
elements of $\mathbb{Z}_n = \{0, 1, \ldots, n-1\}$.

### Arithmetic Rules

Congruence is preserved under addition and multiplication:

$$a \equiv a' \pmod{n} \;\wedge\; b \equiv b' \pmod{n}$$
$$\Rightarrow\; a + b \equiv a' + b' \pmod{n} \;\wedge\; ab \equiv a'b' \pmod{n}$$

This means we can do arithmetic modulo $n$ by taking remainders at each step —
no numbers need grow beyond $n-1$.

### Modular Exponentiation

For large exponents, **fast modular exponentiation** (repeated squaring) computes
$a^e \pmod{n}$ in $O(\log e)$ multiplications rather than $O(e)$:

$$a^e \pmod{n} = \begin{cases}
1 & \text{if } e = 0 \\
(a^{e/2})^2 \bmod n & \text{if } e \text{ even} \\
a \cdot a^{e-1} \bmod n & \text{if } e \text{ odd}
\end{cases}$$

**Why this matters.** RSA encryption computes $m^e \bmod n$ where $e$ is a
2048-bit or larger number. Without fast exponentiation, this would be computationally
infeasible. With it, each encryption takes milliseconds.


## 4 · Modular Inverses and the Extended Euclidean Algorithm

### When Does a Modular Inverse Exist?

The **modular inverse** of $a$ modulo $n$ is the integer $x$ such that:

$$ax \equiv 1 \pmod{n}$$

**Theorem.** $a^{-1} \pmod{n}$ exists if and only if $\gcd(a, n) = 1$.

**Proof** ($\Rightarrow$): If $ax \equiv 1 \pmod{n}$, then $ax - 1 = kn$ for some
integer $k$, so $ax - kn = 1$. By Bézout's identity, $\gcd(a,n)$ divides
$ax - kn = 1$, hence $\gcd(a,n) = 1$. $\square$

**Proof** ($\Leftarrow$): If $\gcd(a,n) = 1$, Bézout gives $as + nt = 1$,
so $as \equiv 1 \pmod{n}$, and $s \bmod n$ is the inverse. $\square$

### Uniqueness of the Modular Inverse

**Theorem.** If $\gcd(a, n) = 1$, the modular inverse is unique in $\mathbb{Z}_n$.

**Proof by contradiction** *(fulfilling the Module 00 promise)*. Suppose $x$
and $y$ are both inverses of $a$ modulo $n$:

$$ax \equiv 1 \pmod{n} \qquad\text{and}\qquad ay \equiv 1 \pmod{n}$$

Then $ax \equiv ay \pmod{n}$, so $a(x-y) \equiv 0 \pmod{n}$, meaning
$n \mid a(x-y)$.

Since $\gcd(a,n) = 1$, by Euclid's lemma: $n \mid (x-y)$, so $x \equiv y \pmod{n}$.

The assumption that two distinct inverses exist leads to $x = y$ in $\mathbb{Z}_n$
— a contradiction with our assumption that they were distinct. $\square$

### The Extended Euclidean Algorithm

The Extended Euclidean Algorithm (EEA) computes $\gcd(a,b)$ along with the
Bézout coefficients $s$ and $t$:

```
extended_gcd(a, b):
  if b == 0: return (a, 1, 0)   # gcd=a, s=1, t=0
  g, s, t = extended_gcd(b, a mod b)
  return (g, t, s - (a//b)*t)
```

When $\gcd(a, n) = 1$, the coefficient $s$ gives $a^{-1} \bmod n$.


## 5 · Fermat's Little Theorem

**Theorem (Fermat's Little Theorem).** If $p$ is prime and $\gcd(a, p) = 1$, then:

$$a^{p-1} \equiv 1 \pmod{p}$$

**Equivalently:** $a^p \equiv a \pmod{p}$ for all $a$ (including when $p \mid a$).

### Proof Sketch

Consider the set $S = \{a, 2a, 3a, \ldots, (p-1)a\}$ modulo $p$.
Since $\gcd(a,p) = 1$, multiplying by $a$ permutes $\{1, 2, \ldots, p-1\}$ —
no two elements of $S$ are congruent mod $p$. Therefore $S \equiv \{1, 2, \ldots, p-1\}
\pmod{p}$.

Multiplying both sides:

$$a^{p-1} \cdot (p-1)! \equiv (p-1)! \pmod{p}$$

Since $\gcd((p-1)!, p) = 1$, we can divide both sides by $(p-1)!$:

$$a^{p-1} \equiv 1 \pmod{p} \qquad \square$$

### Applications

**Fast inverse computation.** Since $a^{p-1} \equiv 1 \pmod{p}$:

$$a \cdot a^{p-2} \equiv 1 \pmod{p} \;\Rightarrow\; a^{-1} \equiv a^{p-2} \pmod{p}$$

For prime moduli (as in many cryptographic systems), the modular inverse is
just modular exponentiation — no need for the EEA.

**Primality testing.** If $a^{n-1} \not\equiv 1 \pmod{n}$ for some $a$, then
$n$ is definitely composite (Fermat witness). This underlies the Miller-Rabin
primality test used to generate RSA primes.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[CS\]

| Number theory concept | Cryptographic role | Used in |
|---|---|---|
| $a \mid b$ (divisibility) | Factor structure of RSA modulus $n = pq$ | RSA key generation |
| $\gcd(a, b)$ | Check if $e$ and $\phi(n)$ are coprime — required for RSA | RSA public key selection |
| Euclidean Algorithm | Efficient GCD computation for key generation | RSA, primality testing |
| $a \equiv b \pmod{n}$ | All crypto operations wrap around at modulus $n$ | Every crypto primitive |
| $a^e \bmod n$ (fast exp) | Encryption ($m^e \bmod n$) and decryption ($c^d \bmod n$) | RSA |
| $a^{-1} \bmod n$ (inverse) | Private key $d = e^{-1} \bmod \phi(n)$ | RSA key generation |
| Uniqueness of inverse | Private key is unique — decryption is unambiguous | RSA correctness |
| Fermat's Little Theorem | $a^{p-2}$ computes inverse for prime $p$ | RSA, Diffie-Hellman |
| Coprimality $\gcd(a,n)=1$ | Necessary condition for secure key parameters | RSA, discrete log |

**The Module 00 payoff.** The proof of modular inverse uniqueness used proof by
contradiction exactly as promised in Module 00: assume two distinct inverses exist,
derive $x = y$, contradicting the assumption. The technique used in the first
notebook of the course reappears here to prove a foundational cryptographic property.

---


## Python: Visualization & Verification

**1 · Euclidean Algorithm Tracer** — implement the Euclidean and Extended
Euclidean algorithms with full step logging; visualise the reduction steps and
verify Bézout's identity.

**2 · Modular Arithmetic Explorer** — compute modular addition/multiplication
tables for small $n$; identify which elements have inverses; visualise
$\mathbb{Z}_n^*$ (the units group).

**3 · Fermat's Little Theorem Verifier** — verify FLT computationally for a
range of primes and bases; implement fast modular exponentiation from scratch;
time it against naive exponentiation to show the $O(\log e)$ speedup.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from math import gcd, log2
import time

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · Euclidean Algorithm — Step-by-Step Tracer

We implement both the standard and Extended Euclidean Algorithms with full step
logging, then verify Bézout's identity for several pairs.


In [ ]:
# ── 1 · Euclidean and Extended Euclidean Algorithm ───────────────────────────

def euclidean_trace(a, b):
    """Euclidean algorithm with step logging. Returns (gcd, steps)."""
    steps = []
    while b != 0:
        q, r = divmod(a, b)
        steps.append((a, b, q, r))
        a, b = b, r
    return a, steps

def extended_gcd(a, b):
    """Extended Euclidean Algorithm. Returns (gcd, s, t) where a*s + b*t = gcd."""
    if b == 0:
        return a, 1, 0
    g, s, t = extended_gcd(b, a % b)
    return g, t, s - (a // b) * t

def mod_inverse(a, n):
    """Modular inverse of a mod n (if it exists)."""
    g, s, _ = extended_gcd(a, n)
    if g != 1:
        return None
    return s % n

# ── Trace examples ────────────────────────────────────────────────────────────
examples = [(252, 105), (35, 64), (17, 3120)]

for a, b in examples:
    g, steps = euclidean_trace(a, b)
    ge, s, t = extended_gcd(a, b)
    print(f"gcd({a}, {b}) — Euclidean Algorithm:")
    for (ai, bi, qi, ri) in steps:
        print(f"  {ai} = {qi} × {bi} + {ri}")
    print(f"  gcd = {g}")
    print(f"  Bézout: {a}×{s} + {b}×{t} = {a*s + b*t}  ✓" if a*s + b*t == g else "  CHECK FAILED")
    inv = mod_inverse(a, b) if gcd(a,b) == 1 else None
    if inv:
        print(f"  {a}⁻¹ mod {b} = {inv}  (verify: {a}×{inv} mod {b} = {(a*inv)%b})")
    print()

# ── Visualise GCD reduction steps ─────────────────────────────────────────────
a_demo, b_demo = 252, 105
_, steps = euclidean_trace(a_demo, b_demo)

fig, ax = plt.subplots(figsize=(10, 4))
step_labels = [f"gcd({s[0]},{s[1]})" for s in steps] + [f"gcd({steps[-1][1]},0)"]
remainders  = [s[0] for s in steps] + [steps[-1][1]]

ax.plot(range(len(remainders)), remainders, color=TS_BLUE, lw=2.5,
        marker='o', markersize=10, zorder=3)
ax.fill_between(range(len(remainders)), remainders, alpha=0.15, color=TS_BLUE)

for i, (lbl, rem) in enumerate(zip(step_labels, remainders)):
    ax.text(i, rem + 3, lbl, ha='center', fontsize=8, color=TS_GRAY)

ax.axhline(gcd(a_demo, b_demo), color=TS_GREEN, lw=1.5, linestyle='--',
           label=f'gcd = {gcd(a_demo, b_demo)}')
ax.set_xticks(range(len(step_labels)))
ax.set_xticklabels([f'Step {i}' for i in range(len(step_labels))], fontsize=9)
ax.set_ylabel('Value')
ax.set_title(f'Euclidean Algorithm — gcd({a_demo}, {b_demo})
Each step reduces the problem; terminates at gcd',
             pad=10, fontsize=11)
ax.legend(fontsize=9)
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

Each step of the Euclidean Algorithm replaces $(a, b)$ with $(b, a \bmod b)$,
strictly decreasing the larger value until it reaches 0. The final non-zero
remainder is the GCD.

Bézout's identity is verified for all examples — the linear combination
$as + bt = \gcd(a,b)$ holds exactly. The modular inverse for the coprime
pair is computed and checked: $17 \times s \equiv 1 \pmod{3120}$.

The value 3120 is not arbitrary — it is $\phi(3233) = (p-1)(q-1)$ for the
small RSA example we will use in Unit 02. Computing $17^{-1} \bmod 3120$
is exactly the RSA private key computation. The tools from this section
are the literal implementation of RSA key generation.


### 2 · Modular Arithmetic Explorer — The Structure of $\mathbb{Z}_n$

We compute the multiplication table for $\mathbb{Z}_n$ for small $n$, identify
which elements have modular inverses (the units $\mathbb{Z}_n^*$), and visualise
how the group structure changes between prime and composite moduli.


In [ ]:
# ── 2 · Modular Arithmetic Explorer ──────────────────────────────────────────

def units(n):
    """Return the set of units (elements with multiplicative inverse) in Z_n."""
    return [a for a in range(1, n) if gcd(a, n) == 1]

def mult_table(n):
    """Compute the n×n multiplication table mod n."""
    return np.array([[i*j % n for j in range(n)] for i in range(n)])

# Show structure for n=7 (prime) and n=12 (composite)
for n, label in [(7, 'ℤ₇ (prime modulus)'),
                 (12, 'ℤ₁₂ (composite modulus)')]:
    u = units(n)
    print(f"{label}:")
    print(f"  Units Z_{n}* = {u}  (|Z_{n}*| = {len(u)})")
    print(f"  Elements WITHOUT inverse: {[a for a in range(1,n) if a not in u]}")
    print()

# ── Visualise multiplication tables and unit structure ────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for col, (n, label) in enumerate([(7, 'ℤ₇ (prime, all non-zero elements invertible)'),
                                    (12, 'ℤ₁₂ (composite, only coprime elements invertible)')]):
    # Multiplication table heatmap
    ax = axes[0, col]
    mt = mult_table(n)
    im = ax.imshow(mt, cmap='Blues', aspect='auto')
    for i in range(n):
        for j in range(n):
            ax.text(j, i, str(mt[i,j]), ha='center', va='center', fontsize=8)
    ax.set_xticks(range(n)); ax.set_xticklabels(range(n))
    ax.set_yticks(range(n)); ax.set_yticklabels(range(n))
    ax.set_title(f'Multiplication table mod {n}
({label})', fontsize=9, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046)

    # Units bar chart
    ax2 = axes[1, col]
    has_inv  = [1 if a in units(n) else 0 for a in range(n)]
    bar_cols = [TS_GREEN if h else TS_RED for h in has_inv]
    ax2.bar(range(n), has_inv, color=bar_cols, edgecolor='white', width=0.7)
    ax2.set_xticks(range(n))
    ax2.set_xticklabels([f'{a}
(gcd={gcd(a,n)})' if a > 0 else '0' for a in range(n)],
                        fontsize=7)
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(['No inverse', 'Has inverse'])
    ax2.set_title(f'Invertible elements in ℤ_{n}
'
                  f'Green = gcd(a,{n})=1 → has inverse; Red = no inverse',
                  fontsize=9, fontweight='bold')

plt.suptitle('Structure of ℤₙ: Prime vs Composite Modulus',
             fontsize=12, fontweight='bold', y=1.01)
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()

print("Key insight:")
print("  For prime p: EVERY non-zero element has an inverse → Z_p* = {1,...,p-1}")
print("  For composite n: only elements coprime to n have inverses")
print("  RSA uses n = pq (product of two primes) precisely because φ(n) = (p-1)(q-1)")
print("  is easy to compute when you know p and q, but hard to compute from n alone.")


**What do you see?**

For $\mathbb{Z}_7$ (prime): every non-zero row in the multiplication table contains
every value $1$ through $6$ exactly once — every non-zero element is invertible.
The units group $\mathbb{Z}_7^* = \{1,2,3,4,5,6\}$ has 6 elements.

For $\mathbb{Z}_{12}$ (composite): the multiplication table has repeated values and
zero entries in many rows. Only elements coprime to 12 — those sharing no factor
with 12 — have inverses. Elements 2, 3, 4, 6, 8, 9, 10 have no inverse because they
share a factor with 12.

**The RSA insight.** RSA chooses $n = pq$ where $p$ and $q$ are large primes.
The units group $\mathbb{Z}_n^*$ has $\phi(n) = (p-1)(q-1)$ elements. The
encryption exponent $e$ must be chosen from this group — coprime to $\phi(n)$ —
so that a private key $d = e^{-1} \bmod \phi(n)$ exists.


### 3 · Fermat's Little Theorem — Verification and Fast Exponentiation

We verify FLT computationally for a range of primes and bases, implement fast
modular exponentiation from scratch, and time it against naive exponentiation
to demonstrate the $O(\log e)$ speedup on a real RSA-scale exponent.


In [ ]:
# ── 3 · Fermat's Little Theorem and Fast Modular Exponentiation ───────────────

# ── Verify FLT: a^(p-1) ≡ 1 (mod p) for prime p, gcd(a,p)=1 ─────────────────
primes = [7, 11, 13, 17, 23, 29, 31, 37]
bases  = [2, 3, 5, 7, 10, 15]

print("Fermat's Little Theorem Verification: a^(p-1) mod p = 1")
print(f"{'':>5}", end='')
for p in primes: print(f"  p={p:>2}", end='')
print()
print("─" * 75)

all_pass = True
for a in bases:
    print(f"a={a:>2}", end='')
    for p in primes:
        if gcd(a, p) == 1:
            result = pow(a, p-1, p)
            ok = (result == 1)
            all_pass = all_pass and ok
            print(f"  {'✓' if ok else '✗':>4}", end='')
        else:
            print(f"  {'N/A':>4}", end='')
    print()

print(f"
All FLT checks passed: {all_pass} ✓")

# ── Implement fast modular exponentiation from scratch ────────────────────────
def fast_pow(base, exp, mod):
    """Fast modular exponentiation: base^exp mod modulus in O(log exp) steps."""
    result, steps = 1, 0
    base = base % mod
    while exp > 0:
        if exp % 2 == 1:
            result = (result * base) % mod
        exp //= 2
        base  = (base * base) % mod
        steps += 1
    return result, steps

def naive_pow(base, exp, mod):
    """Naive modular exponentiation: O(exp) multiplications."""
    result = 1
    for _ in range(exp):
        result = (result * base) % mod
    return result

# Compare on RSA-sized exponent
n_rsa   = 3233          # small RSA modulus (for demo: p=61, q=53)
e_rsa   = 17            # public exponent
m       = 65            # message
c, steps = fast_pow(m, e_rsa, n_rsa)
print(f"
Fast pow: {m}^{e_rsa} mod {n_rsa} = {c}  ({steps} squaring steps)")
print(f"Naive pow: {m}^{e_rsa} mod {n_rsa} = {naive_pow(m, e_rsa, n_rsa)}  (match: ✓)")

# Time comparison on a large exponent
large_exp = 2**20   # about 1 million
large_mod = 10**9 + 7

t0 = time.perf_counter()
r_fast, steps_fast = fast_pow(7, large_exp, large_mod)
t_fast = time.perf_counter() - t0

print(f"
Timing comparison for base=7, exp=2^20={large_exp}, mod={large_mod}:")
print(f"  Fast exponentiation: {t_fast*1000:.3f} ms  ({steps_fast} steps)")

# ── Speedup visualisation ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

exp_range = range(1, 33)
fast_steps  = [int(log2(e)) + 1 for e in exp_range]
naive_steps = list(exp_range)

ax = axes[0]
ax.plot(exp_range, [2**e for e in exp_range], color=TS_RED,
        lw=2.5, label='Naive: $O(e)$ multiplications')
ax.plot(exp_range, [2*e for e in exp_range], color=TS_GREEN,
        lw=2.5, label='Fast: $O(\log e)$ squarings')
ax.set_yscale('log')
ax.set_xlabel('Exponent bit length $\lceil \log_2 e \rceil$')
ax.set_ylabel('Number of multiplications (log scale)')
ax.set_title('Naive vs Fast Modular Exponentiation
Step Count', fontweight='bold', pad=8)
ax.legend(fontsize=9)

ax2 = axes[1]
bit_lengths = list(range(1, 2049, 64))
fast_s  = bit_lengths  # O(log e) = O(bit length)
naive_s = [2**b for b in bit_lengths]

ax2.semilogy(bit_lengths, naive_s, color=TS_RED, lw=2.5, label='Naive steps')
ax2.semilogy(bit_lengths, fast_s,  color=TS_GREEN, lw=2.5, label='Fast steps')
ax2.axvline(2048, color=TS_AMBER, lw=1.5, linestyle='--', label='RSA-2048 exponent size')
ax2.set_xlabel('Exponent bit length')
ax2.set_ylabel('Steps (log scale)')
ax2.set_title('RSA-2048 Scale
Fast exp: 2048 steps | Naive: 2^2048 steps',
              fontweight='bold', pad=8)
ax2.legend(fontsize=9)

fig.patch.set_facecolor('white')
plt.suptitle("Fast Modular Exponentiation — O(log e) vs O(e)",
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"
For RSA-2048: exponent has ~2048 bits")
print(f"  Fast exponentiation: ~2048 multiplications")
print(f"  Naive exponentiation: 2^2048 ≈ 10^616 multiplications → infeasible")
print(f"  Fast exponentiation is what makes RSA practical.")


**What do you see?**

Fermat's Little Theorem is verified for every (prime, base) combination — all ✓.

The timing chart makes the speedup visceral: naive exponentiation requires $2^{2048}$
multiplications for RSA-2048, an astronomically large number. Fast exponentiation
requires exactly 2048 squarings — linear in the bit length. This is not a minor
optimisation; it is the difference between milliseconds and heat-death-of-the-universe.

The step count visualisation shows the crossover: for exponents beyond about $e = 4$,
fast exponentiation is already faster, and the gap widens exponentially with each
additional bit. RSA encryption at 2048 bits takes about 2000 multiplications;
naive evaluation would require more operations than atoms in the observable universe.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. WILSON'S THEOREM
#    Wilson's Theorem states: p is prime iff (p-1)! ≡ -1 (mod p).
#    Verify this computationally for all integers 2 ≤ n ≤ 30.
#    Which values satisfy the congruence? Which do not?
#    This gives a primality test — but why is it not used in practice?
#    (Hint: consider the computational cost of computing (p-1)! mod p for large p.)
#
# 2. CHINESE REMAINDER THEOREM
#    The CRT states: if n₁, n₂, ..., nₖ are pairwise coprime, then the system
#      x ≡ a₁ (mod n₁)
#      x ≡ a₂ (mod n₂)
#      ...
#    has a unique solution mod (n₁×n₂×...×nₖ).
#    Solve: x ≡ 2 (mod 3), x ≡ 3 (mod 5), x ≡ 2 (mod 7).
#    Implement crt(remainders, moduli) using the EEA.
#    The CRT is used in RSA optimisation (CRT-RSA) to speed up decryption by ~4×.
#
# 3. MODULAR SQUARE ROOTS
#    For prime p ≡ 3 (mod 4), the square root of a (mod p) is a^((p+1)/4) mod p.
#    Verify: for p=7, compute √2 mod 7 and check it squares back to 2.
#    The Tonelli-Shanks algorithm generalises this for any prime.
#    Modular square roots appear in elliptic curve cryptography — point
#    decompression reconstructs a full curve point from just its x-coordinate
#    using a modular square root computation.

# Your work here:


---

## Summary

| Concept | Definition | Cryptographic role |
|---|---|---|
| $a \mid b$ | $b = ka$ for some integer $k$ | Factor structure of RSA modulus |
| $\gcd(a,b)$ | Largest divisor of both $a$ and $b$ | Coprimality check for key parameters |
| Euclidean Algorithm | $\gcd(a,b) = \gcd(b, a \bmod b)$ | Efficient GCD in $O(\log \min(a,b))$ steps |
| Bézout's Identity | $as + bt = \gcd(a,b)$ | Foundation of modular inverse computation |
| $a \equiv b \pmod{n}$ | $n \mid (a-b)$ | All crypto wraps at modulus $n$ |
| $a^{-1} \bmod n$ | $ax \equiv 1 \pmod{n}$ | Private key $d = e^{-1} \bmod \phi(n)$ |
| Unique inverse | Proved by contradiction | Decryption is unambiguous |
| Fermat's Little Theorem | $a^{p-1} \equiv 1 \pmod{p}$ | Fast inverse; primality testing |
| Fast modular exp | $O(\log e)$ squarings | RSA encrypt/decrypt in milliseconds |

---

## Up Next

**Module 08 · Unit 02 — Cryptographic Primitives**

We assemble the Unit 01 tools into RSA and Diffie-Hellman. Euler's totient function
$\phi(n)$ generalises Fermat's theorem to composite moduli. RSA key generation is
a direct application of the modular inverse computation from this unit. The RSA
correctness proof uses Euler's theorem. Diffie-Hellman introduces the discrete
logarithm problem — the second great hardness assumption in deployed cryptography.

$\rightarrow$ `modules/module-08/unit-02-crypto-primitives.ipynb`
